#FACT SHIPMENTS AND RETURN CLEANING AND TREANSFORMATION

In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "storageaccount", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

##FACT RETURNS

In [0]:
df = (
    spark.readStream
    .format("delta")
    .table(f"{catalog_name}.bronze.brz_order_returns")
)

In [0]:
# Remove duplicate return records
df = df.dropDuplicates([
    "order_id",
    "order_dt",
    "return_ts"
])

# Data cleaning and standardization
df = (
    df

    # Ensure proper data types
    .withColumn(
        "order_dt",
        F.to_date(F.col("order_dt"))
    )

    .withColumn(
        "return_ts",
        F.to_timestamp(F.col("return_ts"))
    )

    # Standardize return reason
    .withColumn(
        "reason",
        F.upper(F.trim(F.col("reason")))
    )

    # Add processing timestamp
    .withColumn(
        "processed_time",
        F.current_timestamp()
    )
)

In [0]:
# df_static = spark.table(f"{catalog_name}.bronze.brz_order_returns")

# df_static.select([
#     F.count(F.when(F.col(c).isNull(), c)).alias(c)
#     for c in df_static.columns
# ]).show()

+---------+--------+--------+------+-------------+----------------+-----------+
|return_ts|order_dt|order_id|reason|_rescued_data|ingest_timestamp|source_file|
+---------+--------+--------+------+-------------+----------------+-----------+
|        0|       0|       0|     0|        20194|               0|          0|
+---------+--------+--------+------+-------------+----------------+-----------+



In [0]:
silver_checkpoint_path = (
    f"abfss://{container_name}@{storage_account_name}"
    f".dfs.core.windows.net/checkpoint/silver/fact_order_returns/"
)

In [0]:
def upsert_to_silver(microBatchDF, batchId):

    table_name = f"{catalog_name}.silver.slv_order_returns"

    # First run - create table
    if not spark.catalog.tableExists(table_name):

        print("Creating Silver table...")

        (
            microBatchDF.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )

        spark.sql(
            f"""
            ALTER TABLE {table_name}
            SET TBLPROPERTIES
            (delta.enableChangeDataFeed = true)
            """
        )

    # Subsequent runs - upsert
    else:

        deltaTable = DeltaTable.forName(
            spark,
            table_name
        )

        (
            deltaTable.alias("silver_table")
            .merge(
                microBatchDF.alias("batch_table"),
                """
                silver_table.order_id = batch_table.order_id
                AND silver_table.order_dt = batch_table.order_dt
                AND silver_table.return_ts = batch_table.return_ts
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

In [0]:
(
    df.writeStream
      .foreachBatch(upsert_to_silver)
      .option(
          "checkpointLocation",
          silver_checkpoint_path
      )
      .outputMode("update")
      .trigger(availableNow=True)
      .start()
      .awaitTermination()
)

##FACT SHIPMENTS

In [0]:
# Read Bronze stream
df = (
    spark.readStream
    .format("delta")
    .table(f"{catalog_name}.bronze.brz_order_shipments")
)

# Data cleaning and standardization
df = (
    df

    # Ensure proper date type
    .withColumn(
        "order_dt",
        F.to_date(F.col("order_dt"))
    )

    # Standardize carrier names
    .withColumn(
        "carrier",
        F.upper(F.trim(F.col("carrier")))
    )

    # Add processing timestamp
    .withColumn(
        "processed_time",
        F.current_timestamp()
    )
)

In [0]:
silver_checkpoint_path = (
    f"abfss://{container_name}@{storage_account_name}"
    f".dfs.core.windows.net/checkpoint/silver/fact_order_shipments/"
)

def upsert_to_silver(microBatchDF, batchId):

    table_name = f"{catalog_name}.silver.slv_order_shipments"

    # First run - create table
    if not spark.catalog.tableExists(table_name):

        print("Creating Silver table...")

        (
            microBatchDF.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )

        spark.sql(
            f"""
            ALTER TABLE {table_name}
            SET TBLPROPERTIES
            (delta.enableChangeDataFeed = true)
            """
        )

    # Incremental loads
    else:

        deltaTable = DeltaTable.forName(spark, table_name)

        (
            deltaTable.alias("silver_table")
            .merge(
                microBatchDF.alias("batch_table"),
                """
                silver_table.shipment_id = batch_table.shipment_id
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

In [0]:
(
    df.writeStream
      .foreachBatch(upsert_to_silver)
      .option(
          "checkpointLocation",
          silver_checkpoint_path
      )
      .outputMode("update")
      .trigger(availableNow=True)
      .start()
      .awaitTermination()
)

In [0]:
spark.table(f"{catalog_name}.bronze.brz_order_shipments") \
     .select("shipment_id") \
     .distinct() \
     .count()

679731